# SMS Spam Detection

**Goal:** Classify SMS as Spam or Ham (legitimate)
**Algorithm:** Naive Bayes with TF-IDF (best for text)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score)

In [ ]:
np.random.seed(42)
n = 500

ham_messages = [
    'Hey how are you doing today',
    'Want to grab coffee later',
    'See you at the meeting tomorrow',
    'Can you pick up some groceries',
    'Happy birthday have a great day'
]
spam_messages = [
    'CONGRATULATIONS You won a FREE iPhone claim now',
    'URGENT Your account needs verification click here',
    'WIN BIG cash prizes call now 1800',
    'Exclusive offer just for you limited time',
    'You have been selected for a FREE vacation'
]

texts = []
labels = []
# 400 ham, 100 spam (realistic ratio)
for _ in range(400):
    texts.append(np.random.choice(ham_messages))
    labels.append('ham')
for _ in range(100):
    texts.append(np.random.choice(spam_messages))
    labels.append('spam')

df = pd.DataFrame({'message': texts, 'label': labels})
print ('Shape: %s' % (df.shape,))

<hr>## 1. Exploratory Data Analysis

In [ ]:
print ('Label distribution:\n%s' % df['label'].value_counts())
print ('\nSpam ratio: %.2f%%' % (df['label'].value_counts(normalize=True)['spam'] * 100))
print ('\nFirst 5 rows:\n%s' % df.head())

<hr>## 2. Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df['message']).toarray()
y = df['label'].map({'ham': 0, 'spam': 1})

print ('Feature matrix: %s' % (X.shape,))

<hr>## 3. Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print ('Train: %s, Test: %s' % (X_train.shape[0], X_test.shape[0]))

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)
print ('Model: %s' % model)

<hr>## 4. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)

print ('Accuracy:  %.4f' % accuracy_score(y_test, y_pred))
print ('Precision: %.4f' % precision_score(y_test, y_pred))
print ('Recall:    %.4f' % recall_score(y_test, y_pred))
print ('\nConfusion Matrix:\n%s' % confusion_matrix(y_test, y_pred))

In [ ]:
print ('Classification Report:\n%s' % classification_report(
    y_test, y_pred, target_names=['Ham', 'Spam']))

<hr>## 5. Test Custom Messages

In [ ]:
test_messages = [
    'Hey lets meet for lunch tomorrow',
    'FREE MONEY click here to claim your prize NOW'
]

for msg in test_messages:
    vec = vectorizer.transform([msg]).toarray()
    prob = model.predict_proba(vec)[0]
    pred = model.predict(vec)[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    print ("'%s...' -> %s (spam probability: %.2f%%)" % (msg[:30], label, prob[1]*100))